# Blob to Bronze — Auto Loader Ingestion

Ingests CSV files from blob source volumes into bronze Delta tables using Auto Loader with `Trigger.AvailableNow`. Each table has its own source directory, checkpoint, and schema location.

In [0]:
TARGET_CATALOG = "mano_retailq"
TARGET_SCHEMA = "blob_bronze"
CHECKPOINT_ROOT = "/Volumes/mano_retailq/volume/blob_source/_checkpoints"

# Each table has its own source directory, checkpoint, and schema location.
TABLES = {
    "order_items": {
        "source_dir": "/Volumes/mano_retailq/volume/blob_source/orders/",
        "checkpoint": f"{CHECKPOINT_ROOT}/order_items",
    },
    "sales_orders": {
        "source_dir": "/Volumes/mano_retailq/volume/blob_source/sales/",
        "checkpoint": f"{CHECKPOINT_ROOT}/sales_orders",
    },
}

In [0]:
from pyspark.sql.functions import col, current_timestamp


def load_to_bronze(table_name: str, source_dir: str, checkpoint: str) -> None:
    """Run one Auto Loader stream for a single table and wait for it to finish."""
    full_table = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

    raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        # Source CSVs use the literal string NULL for missing values
        .option("nullValue", "NULL")
        # Infer exact column types instead of defaulting to strings
        .option("cloudFiles.inferColumnTypes", "true")
        # Auto Loader stores the inferred schema under <checkpoint>/_schemas/
        .option("cloudFiles.schemaLocation", checkpoint)
        .load(source_dir)
    )

    bronze_df = raw.withColumns({
        "source_file": col("_metadata.file_name"),
        "ingested_at": current_timestamp(),
    })

    query = (
        bronze_df.writeStream
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(full_table)
    )
    query.awaitTermination()
    print(f"Loaded {table_name} -> {full_table}")


for table_name, spec in TABLES.items():
    load_to_bronze(table_name, spec["source_dir"], spec["checkpoint"])